# Projet 07 - IA agentique : Créer un Agent IA qui planifie tes vacances

*Temps estimé : 120 minutes*

*Difficulté : 6/10*

<div align="center">
<img src="./image.png" alt="ouvrir dans VSCode pour voir l'image" width="800"/>
</div>

Tu es monté dans la voiture. Léa conduit, Marc est devant, le directeur de l'aéroport est assis à côté de toi. Personne ne parle pendant trois minutes.

Puis le directeur sort la clé USB de sa poche.

« Voilà l'idée. Chacun de nous a des données que les deux autres n'ont pas. Moi, j'ai tous les vols au départ de Nice et de Paris. Marc a les brochures de son réseau d'hôteliers. Léa a la liste des activités de chaque ville. Séparément, ça ne vaut rien. »

Marc se retourne. « Ce qu'on veut, c'est quelque chose à qui on dit : *Madrid, du 12 au 16 août, 1000 euros, on aime les musées*, et qui nous propose le voyage complet. Le vol, l'hôtel, le programme, le prix. Et si le budget ne suffit pas, qu'il se débrouille pour trouver un compromis intelligent. Et au bout, qu'il réserve. »

Tu regardes la clé. Trois fichiers, trois métiers, et un problème qu'aucun des trois ne peut résoudre seul.

C'est le dernier projet de l'été. On va construire un **agent**.

## Qu'est-ce qu'un Agent IA ?

De nos jours, on entend partout le terme d'« agent IA », et malheureusement, celui-ci est presque toujours mal expliqué.

Alors mettons les choses au clair tout de suite : **Un agent, c'est en fait une boucle.**

> Il part d'un objectif → puis sélectionne un **outil** pour obtenir des résultats → il évalue le résultat → et **corrige** si ça ne remplit pas l'objectif (autrement dit, il reprend la boucle)

**Un Agent, ce n'est donc PAS un LLM.**... (D'ailleurs, dans ce projet, nous n'utiliserons aucun LLM)

Cependant, des LLMs peuvent être utilisés pour booster l'efficacité de l'agent : Par exemple, traduire la demande d'un utilisateur en objectif : "je veux aller à Madrid pendant 2 semaines avec ma famille, visiter la ville et profiter des meilleurs musées, nous avons 4000 euros" -> Le LLM extrait les informations `Madrid`, `deux semaines`, `familles` (ils demandera sans doute ensuite combien de personnes?), `meilleurs musées`, `budget: 4000`...

Mais retiens bien que ce qui fait l'agent, ça n'est pas le LLM, mais ce sont ses outils et sa capacité à se rattraper.

Un programme qui appelle trois fonctions à la suite, ce n'est pas un agent, c'est un script. En revanche, un programme qui constate que son plan dépasse ton budget vacances, qui choisit lui-même quoi sacrifier, qui recalcule, qui t'explique ensuite ce qu'il a fait, et qui finit par réserver ton avion... ça, c'est un agent.


## Ce qu'on va construire ici

Avant d'écrire la moindre ligne de code, regardons ensemble l'architecture complète de ce projet.

<div align="center">
<img src="./architecture.png" alt="ouvrir dans VSCode pour voir le schéma" width="900"/>
</div>

Prenons une minute pour lire ce schéma, de gauche à droite :

- **Toi**, l'utilisateur, tu exprimes une demande : une destination, des dates, un budget, des envies. Comme je l'ai dit juste avant, pour garder les choses simples, nous n'utiliserons pas de LLMs pour cette tâche (autrement le temps de calcul serait trop long). À la place, nous allons mettre en place des "champs" à remplir (comme pour un vrai site de booking) et l'agent utilisera ces champs pour construire son voyage. Cependant, par la suite, rien ne t'empêchera de continuer ce projet en y intégrant un LLM comme nous l'avons vu dans le projet 04, ou bien en utilisant une API payante (Claude ou ChatGPT par exemple)


- **L'agent**, au centre, reçoit cette demande. Il ne connaît rien par lui-même : tout ce qu'il sait, il va le chercher via ses trois **outils**.


- Deux outils interrogent la **base SQL** (les vols et les activités), le troisième fouille les **brochures PDF** des hôtels grâce à un **modèle d'embeddings**. Et bonne nouvelle... dans ce projet, les outils de notre agent, tu les connais déjà ! Il s'agit en effet : 
    - **du SQL du Projet 01**, pour les vols et les activités. Ce sont des données structurées, avec des prix et des dates, qu'on veut filtrer et trier.
    - **des embeddings du Projet 04**, pour les hôtels. Ce sont des brochures en texte libre, dans lesquelles on cherche « un hôtel calme avec une piscine ». Aucune requête SQL ne sait faire ça.


- Une fois les informations réunies, l'agent entre dans sa **boucle** : il chiffre le voyage, vérifie le budget, et si ça ne rentre pas, il sacrifie quelque chose et recommence.


- À la fin, il te rend le voyage, la liste de ses choix, et il peut réserver ton vol d'avion directement dans la base de données.

Et la cerise sur le gâteau, pour ce dernier projet du cahier de vacances, nous verrons également comment construire une **Interface Web** afin d'utiliser notre Agent ailleurs que dans un notebook ou bien notre terminal.

Let's Go ! 

## Import de nos librairies

Comme toujours, j'ai créé une série de fonctions dans le module `utils`, afin de gagner du temps dans ce projet. Ces fonctions reprennent ce que nous avons utilisé dans les précédents projets (`encoder` pour la matrice d'embedding, `requete` pour nos requêtes SQL...)

In [ ]:
from utils import (afficher_hotels, afficher_voyage, afficher_vols, charger_brochures,
                   charger_encodeur, connexion, encoder, requete, texte_proposition)

conn = connexion()
brochures = charger_brochures()
encodeur = charger_encodeur()

print(f"{len(brochures)} brochures d'hôtels chargées")

# Partie 1 - Le chargement des données

Léa, Marc et le directeur de l'aéroport t'ont confié 3 clés USB. Voyons ce qui se cache dessus.

In [ ]:
print("=== Les tables de la base de données ===")
display(requete(conn, "SELECT name FROM sqlite_master WHERE type='table'"))

In [ ]:
print("=== Un aperçu de la table `vols` ===")
display(requete(conn, "SELECT * FROM vols LIMIT 3"))

In [ ]:
print("=== Un aperçu de la table `activités` ===")
display(requete(conn, "SELECT * FROM activites LIMIT 3"))


print("=== Et un exemple des activités de Barcelone ===")
display(requete(conn, "SELECT * FROM activites WHERE ville = 'Barcelone'"))

In [ ]:
print("=== Les brochures d'hôtels (des PDF, comme au Projet 04) ===")
display(brochures[["ville", "hotel", "etoiles", "note", "avis", "prix_nuit"]].head(12))
print(f"({len(brochures)} brochures en tout, {brochures['ville'].nunique()} villes)")

Nous avons donc une base de données comprenant des **vols** et des **activités**...

...Ainsi que des **hôtels**, qui eux, sont des `brochures` rédigées en français, fournies en PDF par le réseau de Marc.

In [ ]:
print(brochures[brochures["ville"] == "Madrid"].iloc[0]["texte"])

# Partie 2 - Le premier outil : Le sélectionneur de vols d'avion

Nous allons maintenant créer notre premier outil.

Un outil, pour un agent, c'est une fonction qui va chercher un fait précis dans le monde extérieur. On va maintenant construire le premier : `outil_vols`, qui interroge la base de l'aéroport et répond à la question « quels avions partent vers cette ville, ce jour-là, et à quel prix ? ».

Pour cela, on va écrire une requête SQL, comme au Projet 01. Mais il y a une nouveauté par rapport au début de l'été, et je te la présente correctement parce qu'elle est importante.

Au Projet 01, tes requêtes étaient écrites en entier, avec les valeurs directement dedans : `WHERE origin = 'Nice'`. Ici, c'est différent : la destination et la date arrivent en **paramètres de la fonction**, et elles changent à chaque appel. La tentation naturelle serait de fabriquer la requête avec un f-string :

```python
sql = f"SELECT * FROM vols WHERE destination = '{destination}'"   # surtout pas !
```

Ça semble fonctionner... jusqu'au jour où ça casse. D'abord, si la valeur contient une apostrophe (imagine une ville qui s'appelle `L'Escala`), l'apostrophe ferme la chaîne SQL au mauvais endroit et la requête devient invalide. Ensuite, et c'est bien plus grave : si la valeur vient d'un utilisateur, il peut y glisser du code SQL qui sera exécuté tel quel. Ça s'appelle une **injection SQL**, et c'est l'une des failles de sécurité les plus exploitées du web depuis 25 ans.

La solution standard est très simple : on écrit un point d'interrogation `?` à la place de chaque valeur, et on fournit les valeurs à part, dans un tuple. La base remplace elle-même chaque `?` par la valeur correspondante, dans l'ordre, sans jamais l'interpréter comme du code. Concrètement :

```python
requete(conn, "SELECT * FROM vols WHERE destination = ? AND date_depart = ?", ("Madrid", "2026-08-12"))
```

Ici, le premier `?` reçoit `"Madrid"` et le deuxième reçoit `"2026-08-12"`. C'est cette forme qu'on utilisera partout dans ce projet, et c'est celle que tu retrouveras dans tous tes futurs projets.

### Étape 1 : chercher les vols

À toi de jouer. Dans la cellule suivante, la fonction est prête, il ne manque que la requête SQL. Elle doit :

1. retourner les colonnes `numero`, `origine`, `heure_depart`, `duree_h`, `prix_eur` et `places_restantes`
2. garder uniquement les vols de la bonne `destination` ET de la bonne `date_depart`
3. vérifier qu'il reste au moins autant de `places_restantes` que de voyageurs
4. trier le tout du moins cher au plus cher, avec `ORDER BY`

Les trois `?` de ta requête seront remplis, dans l'ordre, par le tuple `(destination, date_depart, voyageurs)` que tu vois déjà écrit sur la dernière ligne de la fonction.

In [ ]:
def outil_vols(conn, destination, date_depart, voyageurs=1):
    """
    Outil : trouve les vols disponibles vers une destination, un jour donné.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    destination -- le nom de la ville, tel qu'écrit dans la table
    date_depart -- le jour du départ, au format "2026-08-12"
    voyageurs -- le nombre de places nécessaires

    Retourne :
    vols -- un DataFrame des vols trouvés, du moins cher au plus cher
    """

    ### START CODE HERE ###

    sql = """
        
    """  # (1) les colonnes, (2) les filtres, (3) les places, (4) le tri

    ### END CODE HERE ###

    return requete(conn, sql, (destination, date_depart, voyageurs))


vols_madrid = outil_vols(conn, "Madrid", "2026-08-12", voyageurs=2)
print(f"{len(vols_madrid)} vol(s) pour Madrid le 12 août :")
afficher_vols(vols_madrid)

assert not vols_madrid.empty, "On attend au moins un vol pour Madrid ce jour-là"
assert list(vols_madrid.columns) == ["numero", "origine", "heure_depart", "duree_h",
                                      "prix_eur", "places_restantes"], \
    "Les colonnes attendues sont numero, origine, heure_depart, duree_h, prix_eur, places_restantes"
assert vols_madrid["prix_eur"].is_monotonic_increasing, "Les vols doivent être triés du moins cher au plus cher"
assert (vols_madrid["places_restantes"] >= 2).all(), "Il faut au moins 2 places dans chaque vol retourné"
print("\nExercice validé !")

Le deuxième outil SQL, `outil_activites`, est construit exactement sur le même modèle : il liste ce qu'il y a à faire dans une ville, de la sortie la moins chère à la plus chère.

À toi de jouer ! Retourne `nom, categorie, duree_h, prix_eur` pour une ville sélectionnée

In [ ]:
def outil_activites(conn, ville):
    """
    Outil : liste les activités proposées dans une ville, de la moins chère à la plus chère.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    ville -- le nom de la ville

    Retourne :
    activites -- un DataFrame des activités de la ville
    """

    ### START CODE HERE ###

    # (1) les colonnes, (2) le filtre sur la ville, (3) le tri de la moins chère à la plus chère
    sql = ""

    ### END CODE HERE ###

    return requete(conn, sql, (ville,))


activites_rome = outil_activites(conn, "Rome")
display(activites_rome)

attendu = requete(conn, "SELECT COUNT(*) AS n FROM activites WHERE ville = 'Rome'").iloc[0]["n"]

assert list(activites_rome.columns) == ["nom", "categorie", "duree_h", "prix_eur"], \
    "Les colonnes attendues sont nom, categorie, duree_h et prix_eur, dans cet ordre"
assert len(activites_rome) == attendu, \
    f"On attend les {attendu} activités de Rome, tu en retournes {len(activites_rome)} : " \
    "as-tu bien filtré sur la ville ?"
assert activites_rome["prix_eur"].is_monotonic_increasing, \
    "Les activités doivent être triées de la moins chère à la plus chère"
print("\nExercice validé !")


# Partie 3 - L'outil des hôtels

Passons maintenant au problème le plus intéressant du projet. Le voyageur écrit, avec ses mots à lui :

> « un hôtel pour la famille avec une piscine, et des visites dans la ville »

Ni « famille » ni « piscine » ne sont des colonnes de notre tableau. Ces informations sont enfouies dans le texte des brochures, chacune avec son propre vocabulaire : l'une parle d'un établissement « où les enfants sont vraiment les bienvenus », l'autre d'un « havre de calme ». Une recherche par mot-clé passerait à côté de la moitié des bons hôtels.

Avant de résoudre ce problème, regardons de plus près ce que la fonction `charger_brochures` (fournie dans `utils.py`) a préparé pour nous :

In [ ]:
print("Les colonnes du tableau brochures :", list(brochures.columns))

exemple = brochures[brochures["ville"] == "Madrid"].iloc[0]
print(f"\nNotre hôtel d'exemple : {exemple['hotel']}, {exemple['etoiles']} étoiles, note {exemple['note']}/10")

print("\n=== Sa colonne resume : la présentation et les avis clients, rien d'autre ===")
print(exemple["resume"])

La fonction a donc lu chacun des 135 PDF et en a tiré une ligne par hôtel. Les colonnes chiffrées (`etoiles`, `note`, `prix_nuit`) ont été extraites automatiquement du texte. Et pour le texte lui-même, tu vois qu'il existe **deux colonnes** :

- **`texte`** contient le document complet : la présentation, la liste d'équipements, les avis clients et le pied de page.
- **`resume`** ne garde que la présentation et les avis clients.

Pourquoi cette deuxième colonne ? C'est une leçon importante, alors prenons le temps de l'expliquer. La liste d'équipements (« wifi gratuit, climatisation, petit-déjeuner buffet... ») et le pied de page sont **presque identiques d'une brochure à l'autre**. Si on transforme le document complet en vecteur, tout ce texte commun tire les vecteurs les uns vers les autres, et les hôtels deviennent difficiles à distinguer. En ne gardant que la présentation et les avis, là où s'exprime vraiment le caractère de chaque hôtel, la recherche devient nettement plus précise. J'ai fait la mesure sur nos 135 brochures, avec sept demandes types par ville : en encodant le document complet, le bon hôtel arrive en tête 80 % du temps. En n'encodant que la présentation et les avis, on monte à 92 %.

Retiens l'idée générale : **on ne donne pas tout au modèle, on lui donne ce qui porte le sens.** C'est une décision que tu prendras dans chaque projet de ce type.

### La recherche par le sens

On va maintenant écrire la fonction `outil_hotels`, qui reçoit une ville et une envie écrite librement, et qui retourne les brochures qui s'en rapprochent le plus. Pour cela, on reprend exactement la technique du Projet 04 : transformer les textes en vecteurs, puis comparer les vecteurs. Deux textes qui parlent de la même chose avec des mots différents donnent des vecteurs proches, et c'est ce qui va nous sauver.

La première ligne de la fonction est déjà écrite pour toi : elle filtre le tableau `brochures` pour ne garder que les hôtels de la ville demandée, dans une variable qui s'appelle `hotels_ville`. C'est sur ce petit tableau (9 hôtels environ) que tu vas travailler, en quatre temps :

1. encode les résumés des brochures : `encoder(encodeur, hotels_ville["resume"])` te renvoie une matrice avec une ligne par hôtel
2. encode l'envie du voyageur : attention, `encoder` attend une **liste** de textes, donc on écrit `encoder(encodeur, [envie])` puis on prend `[0]` pour récupérer l'unique vecteur
3. calcule le score de chaque hôtel et range-le dans une nouvelle colonne du tableau, qui doit s'appeler `score` : `hotels_ville["score"] = vecteurs @ vecteur_envie`. Les vecteurs sortent de `encoder` déjà normalisés (longueur 1), donc ce simple produit est exactement la **similarité cosinus** que tu as vue au Projet 04
4. trie par score décroissant et garde les `k` premiers : `sort_values`, puis `head(k)`, puis `reset_index(drop=True)` pour repartir d'un index propre

In [ ]:
def outil_hotels(brochures, encodeur, ville, envie, k=3):
    """
    Outil : trouve les hôtels d'une ville qui correspondent le mieux à une envie écrite librement.

    Arguments :
    brochures -- le DataFrame retourné par charger_brochures
    encodeur -- le modèle retourné par charger_encodeur
    ville -- le nom de la ville
    envie -- ce que cherche le voyageur, avec ses mots à lui
    k -- le nombre d'hôtels à retourner

    Retourne :
    hotels -- un DataFrame des k meilleures brochures, avec une colonne "score", les meilleures d'abord
    """
    hotels_ville = brochures[brochures["ville"] == ville].reset_index(drop=True)

    ### START CODE HERE ###

    vecteurs = # (1) une ligne par brochure de la ville
    vecteur_envie = # (2) un seul vecteur pour la demande
    hotels_ville["score"] = # (3) vecteurs normalisés : produit scalaire = cosinus
    hotels = # (4)

    ### END CODE HERE ###

    return hotels


ENVIE = "un hôtel pour la famille avec une piscine, et des visites dans la ville"

for envie in [ENVIE, "je veux faire la fête et sortir le soir"]:
    print(f'« {envie} »')
    afficher_hotels(outil_hotels(brochures, encodeur, "Madrid", envie))
    print()

famille = outil_hotels(brochures, encodeur, "Madrid", "en famille avec des enfants")
piscine = outil_hotels(brochures, encodeur, "Madrid", "une piscine pour nager")

assert "score" in famille.columns, "Il manque la colonne score"
assert len(famille) == 3, "On attend k = 3 hotels"
assert famille["score"].is_monotonic_decreasing, "Les hotels doivent etre tries du plus au moins pertinent"
assert any(mot in famille.loc[0, "texte"].lower() for mot in ("famille", "enfants")), \
    f"Pour la famille on attend un hotel familial, tu obtiens {famille.loc[0, 'hotel']}"
assert "piscine" in piscine.loc[0, "texte"].lower(), \
    f"Pour une piscine on attend un hotel avec piscine, tu obtiens {piscine.loc[0, 'hotel']}"
print("Exercice validé ! Le même modèle, deux demandes, deux hôtels opposés.")

Prends une seconde pour regarder les scores : ils ne valent pas 0 ou 1, ils sont continus. C'est une recherche par **proximité de sens**, pas une correspondance exacte, et c'est pour ça qu'elle fonctionne sur des phrases écrites librement.

Regarde aussi les deuxième et troisième hôtels du trio. Ils ne sont pas familiaux, mais ils ont une piscine : le modèle a compris que la demande portait sur plusieurs critères, et il propose ceux qui en satisfont une partie. Ce n'est pas un défaut, c'est exactement pour ça qu'on retient trois hôtels et pas un seul : quand le budget se resserrera, l'agent aura besoin de solutions de repli qui ressemblent encore à ce qu'on lui a demandé.


> Note Importante : Notre modèle d'embedding ne comprend pas la `negation`. De ce fait, si l'on écrit "Hôtel Calme réservé aux amoureux, sans enfant" -> il proposera sûrement des hôtels pour les familles avec enfants ! C'est l'une des limites des Embeddings, et la raison pour laquelle les LLMs sont justement utilisés dans la pratique afin de comprendre le vrai sens d'un message ! Comme énoncé en introduction, tu pourras toi-même explorer cela en intégrant le modèle `Qwen` du projet 04 pour aller plus loin suite à ce notebook.

# Partie 4 - Le cerveau de l'agent : décider

Notre agent sait maintenant chercher. Il faut qu'il apprenne à choisir, et surtout à se débrouiller quand tout ne rentre pas dans le budget.

## Juste avant : Deux fonctions que je te donne

Avant d'attaquer le coeur du projet, il nous faut deux petites fonctions utilitaires. Elles ne sont pas des exercices, je te les donne, mais lis-les : la seconde va servir de fondation à quelque chose d'intéressant.

**`chiffrer_voyage`** calcule le prix d'un voyage avant qu'on le propose : le vol, plus l'hôtel multiplié par le nombre de nuits, plus les activités retenues. Une convention pour garder les calculs simples : **tout est compté par personne**, l'hôtel aussi. Le budget du voyageur est donc un budget par personne.

**`dates_voisines`** retourne les jours situés autour de la date demandée : la veille, le lendemain, puis l'avant-veille et le surlendemain. Pourquoi diable en aurions-nous besoin ? Patience, c'est l'Étape 4.

In [ ]:
from datetime import datetime, timedelta


def chiffrer_voyage(vol, hotel, activites, nuits):
    """
    Calcule le prix total d'un voyage, par personne.

    Arguments :
    vol -- une ligne du DataFrame des vols
    hotel -- une ligne du DataFrame des hôtels
    activites -- une liste de dictionnaires, chacun avec une clé "prix_eur"
    nuits -- le nombre de nuits

    Retourne :
    total -- le prix total en euros, par personne
    """
    prix_vol = vol["prix_eur"]
    prix_hotel = hotel["prix_nuit"] * nuits
    prix_activites = sum(a["prix_eur"] for a in activites)
    return float(prix_vol + prix_hotel + prix_activites)


def dates_voisines(date_depart, ecart=2):
    """
    Les jours à explorer autour de la date demandée, du plus proche au plus lointain.

    Arguments :
    date_depart -- le jour souhaité, au format "2026-08-12"
    ecart -- de combien de jours on accepte de s'éloigner

    Retourne :
    voisines -- la liste des autres dates, sans la date demandée elle-même
    """
    jour = datetime.strptime(date_depart, "%Y-%m-%d")
    voisines = []
    for decalage in range(1, ecart + 1):
        for signe in (-1, 1):
            voisines.append((jour + timedelta(days=signe * decalage)).strftime("%Y-%m-%d"))
    return voisines


vol_test = vols_madrid.iloc[0]
hotel_test = outil_hotels(brochures, encodeur, "Madrid", ENVIE).iloc[0]
activites_test = outil_activites(conn, "Madrid").to_dict("records")

print(f"Vol {vol_test['numero']} : {vol_test['prix_eur']:.0f} EUR")
print(f"{hotel_test['hotel']} x 4 nuits : {hotel_test['prix_nuit'] * 4:.0f} EUR")
print(f"{len(activites_test)} activités : {sum(a['prix_eur'] for a in activites_test):.0f} EUR")
print(f"TOTAL : {chiffrer_voyage(vol_test, hotel_test, activites_test, 4):.0f} EUR")
print(f"\nAutour du 12 août, l'agent pourra aussi regarder : {dates_voisines('2026-08-12')}")

## Et enfin... la boucle de l'agent

Nous y voilà. C'est le coeur du projet, la partie qui transforme nos trois outils en un véritable agent.

Le premier plan que l'agent essaie est toujours le même : le vol le moins cher, l'hôtel le plus pertinent, et toutes les activités de la ville. La plupart du temps, ce plan idéal dépasse le budget. Un simple script s'arrêterait là en répondant « impossible ». Notre agent, lui, va **relâcher une contrainte et réessayer**, autant de fois que nécessaire. Et à chaque sacrifice, il le note dans un journal, pour pouvoir tout t'expliquer à la fin.

Reste à décider ce qu'on sacrifie en premier, et cet ordre n'a rien d'anodin :

1. **D'abord les activités payantes**, en commençant par la plus chère. Le voyageur en a beaucoup, on peut en retirer. Remarque bien le mot « payantes » : retirer une visite gratuite ne ferait économiser rien du tout, ce serait du gâchis pur.
2. **Ensuite seulement l'hôtel**, en descendant vers des options moins chères. Pourquoi en dernier ? Parce que l'hôtel, le voyageur l'a explicitement décrit (« pour la famille, avec une piscine »). Sacrifier en premier ce qui a été demandé en premier, c'est la meilleure façon de le décevoir.
3. S'il n'y a plus rien à relâcher, l'agent avoue que c'est impossible. On y reviendra, c'est important.

Tu verras aussi apparaître un `print` au milieu de la boucle. Il n'a aucun rôle dans le calcul : il affiche dans la console une ligne par combinaison essayée, pour que tu voies l'agent chercher au lieu de le deviner. Garde un oeil dessus, c'est le meilleur moyen de comprendre ce qui se passe.

Dans la cellule suivante, la mécanique autour de la boucle est déjà en place, et deux variables méritent qu'on s'y arrête avant que tu écrives quoi que ce soit :

- **`ordre`** est la liste des hôtels dans l'ordre où on les essaiera. Elle contient des numéros de lignes du tableau `hotels` : d'abord le `0`, l'hôtel le plus pertinent, puis uniquement ceux qui coûtent réellement moins cher que lui, rangés du plus cher au moins cher pour descendre en douceur plutôt que de tomber d'un coup sur le moins cher de tous. Au passage, le `key=lambda j: hotels.iloc[j]["prix_nuit"]` que tu verras dans le code signifie simplement « trie ces numéros selon le prix de l'hôtel correspondant ».
- **`position`** indique où on en est dans cette liste. Elle démarre à 0, et l'hôtel en cours d'essai est donc `hotels.iloc[ordre[position]]` : la ligne du tableau dont le numéro est à la position courante de la liste.

Maintenant que tu sais lire ces deux variables, regarde la structure de la boucle : les trois cas sont déjà posés (`if`, `elif`, `else`), et la phrase du journal est déjà écrite pour chacun. Il te reste à remplir les trois petites zones d'exercice, cinq lignes en tout.

**Premier cas, `if payantes:`** (il reste des activités payantes au programme, et la liste `payantes` est déjà calculée pour toi) :

1. `plus_chere` : la sortie la plus chère de `payantes`. Écris `max(payantes, key=lambda a: a["prix_eur"])` (là aussi, le `key=` veut dire « compare selon le prix »)
2. `retenues` : la même liste, sans elle. Le plus propre est `[a for a in retenues if a is not plus_chere]`
3. `position` : remets-la à 0, pour repartir du meilleur hôtel maintenant que le programme est moins cher

**Deuxième cas, `elif position + 1 < len(ordre):`** (plus d'activité payante à sacrifier, mais il reste un hôtel moins cher à essayer) :

4. `position` : avance d'un cran dans la liste `ordre`

**Troisième cas, `else:`** (il n'y a plus rien du tout à relâcher) :

5. `impossible` : passe-la à `True`, et la fonction s'arrêtera juste en dessous en retournant `None`

In [ ]:
def essayer_une_date(demande, conn, brochures, encodeur):
    """
    Compose le meilleur voyage possible pour UN jour de départ donné.

    Arguments :
    demande -- un dictionnaire avec destination, date_depart, nuits, voyageurs, budget_max, envie
    conn, brochures, encodeur -- les sources de données de l'agent

    Retourne :
    voyage -- un dictionnaire décrivant le voyage, ou None si rien ne rentre dans le budget
    journal -- la liste des ajustements faits par l'agent, une phrase par sacrifice
    """
    vols = outil_vols(conn, demande["destination"], demande["date_depart"], demande["voyageurs"])
    hotels = outil_hotels(brochures, encodeur, demande["destination"], demande["envie"])
    activites = outil_activites(conn, demande["destination"])

    journal = []
    if vols.empty or hotels.empty:
        return None, ["aucun vol ou aucun hôtel disponible pour cette destination et cette date"]

    vol = vols.iloc[0]                              # le vol le moins cher
    retenues = activites.to_dict("records")         # toutes les activités, pour commencer
    impossible = False

    # L'ordre dans lequel on essaiera les hôtels : le plus pertinent d'abord, puis ceux qui
    # coûtent vraiment moins cher que lui, du plus cher au moins cher, pour descendre en douceur.
    prix_du_premier = hotels.iloc[0]["prix_nuit"]
    replis = [j for j in range(1, len(hotels)) if hotels.iloc[j]["prix_nuit"] < prix_du_premier]
    ordre = [0] + sorted(replis, key=lambda j: hotels.iloc[j]["prix_nuit"], reverse=True)
    position = 0                                    # où on en est dans la liste ordre

    for _ in range(30): # garde-fou anti-boucle infinie
        hotel = hotels.iloc[ordre[position]]
        total = chiffrer_voyage(vol, hotel, retenues, demande["nuits"])

        # Une ligne par combinaison essayée, pour voir l'agent chercher.
        ecart = total - demande["budget_max"]
        verdict = "OK" if ecart <= 0 else f"dépasse de {ecart:.0f}"
        print(f"   {demande['date_depart'][8:]}/{demande['date_depart'][5:7]}  {hotel['hotel'][:20]:20s} "
              f"vol {vol['prix_eur']:4.0f} + hôtel {hotel['prix_nuit'] * demande['nuits']:5.0f} "
              f"+ sorties {sum(a['prix_eur'] for a in retenues):4.0f} = {total:6.0f} EUR   {verdict}")

        if total <= demande["budget_max"]:
            break                                   # le plan tient dans le budget : on s'arrête là

        payantes = [a for a in retenues if a["prix_eur"] > 0]

        if payantes: # repli n°1 : sacrifier une sortie
            ### START CODE HERE ###
            plus_chere = # (1) la sortie la plus chère
            retenues = # (2) on la retire du programme
            position = # (3) on repart du meilleur hôtel
            ### END CODE HERE ###
            journal.append(f"j'ai retiré « {plus_chere['nom']} » ({plus_chere['prix_eur']:.0f} EUR)")

        elif position + 1 < len(ordre):             # repli n°2 : descendre d'un hôtel
            ### START CODE HERE ###
            position = # (4) l'hôtel suivant de la liste
            ### END CODE HERE ###
            journal.append(f"{hotel['hotel']} restait trop cher, "
                           f"j'ai pris {hotels.iloc[ordre[position]]['hotel']} à la place")

        else: # plus rien à relâcher
            ### START CODE HERE ###
            impossible = # (5) l'agent renonce
            ### END CODE HERE ###
            journal.append("je n'avais plus rien à sacrifier ce jour-là")

        if impossible:
            return None, journal

    voyage = {"destination": demande["destination"], "date_depart": demande["date_depart"],
              "nuits": demande["nuits"], "voyageurs": demande["voyageurs"],
              "vol": vol, "hotel": hotels.iloc[ordre[position]], "activites": retenues,
              "prix_total": total, "budget_max": demande["budget_max"]}
    return voyage, journal


demande = {"destination": "Barcelone", "date_depart": "2026-08-12", "nuits": 4, "voyageurs": 2,
           "budget_max": 750, "envie": ENVIE}

voyage, journal = essayer_une_date(demande, conn, brochures, encodeur)
print()
afficher_voyage(voyage, journal)

gratuites = [a["nom"] for a in outil_activites(conn, "Barcelone").to_dict("records")
             if a["prix_eur"] == 0]
gardees = [a["nom"] for a in voyage["activites"]]

assert voyage is not None, "Un voyage devrait être trouvé pour cette demande"
assert voyage["prix_total"] <= demande["budget_max"], "Le voyage retenu doit tenir dans le budget"
assert all(nom in gardees for nom in gratuites), \
    "Retirer une sortie gratuite ne fait rien économiser : l'agent ne doit sacrifier que du payant"

# Une deuxième demande, pour voir le repli n°2 : un palace avec un budget trop juste.
palace = dict(demande, envie="le grand luxe, un palace avec spa", budget_max=950)
voyage_palace, journal_palace = essayer_une_date(palace, conn, brochures, encodeur)
print()
afficher_voyage(voyage_palace, journal_palace)

assert any("à la place" in ligne for ligne in journal_palace), \
    "Sur cette demande, l'agent doit finir par changer d'hôtel"
print("\nExercice validé !")

Regarde bien la deuxième demande. Pour offrir un palace, l'agent a sacrifié **tout** le programme payant, puis il a changé d'hôtel. Le voyageur se retrouve avec un très bel hôtel et plus rien à faire de la journée.

Est-ce le bon comportement ? C'est discutable, et c'est la conséquence directe de l'ordre qu'on a choisi tout à l'heure : les activités d'abord, l'hôtel en dernier. Un agent plus malin remarquerait qu'après avoir changé d'hôtel, le budget permet de remettre deux ou trois sorties au programme.

Retiens surtout ceci : **cet arbitrage n'est écrit nulle part dans les données, c'est nous qui l'avons défini.** Les trois lignes que tu viens d'écrire encodent une opinion sur ce qui compte le plus dans un voyage. C'est le genre de choix qu'on te demandera de défendre en réunion, et c'est très souvent la partie la plus importante du travail.

> Suite à ce projet, je t'invite à modifier toi-même les règles qu'on vient d'écrire pour tenter d'autres approches. S'amuser à démonter le code puis le remonter, afin d'obtenir le résultat qui te plaît vraiment :) 

### Améliorer l'Agent : chercher là où on ne lui a rien demandé

Ton agent sait composer le meilleur voyage pour **le jour qu'on lui a donné**. Mais un vrai bon agent de voyage ferait autre chose : il regarderait aussi les jours d'à côté.

Et il aurait raison. Dans la base de l'aéroport, le prix d'un vol varie fortement d'un jour à l'autre, parce que le week-end coûte plus cher que le milieu de semaine. Personne ne demande jamais « et si je partais un jour plus tôt ? », tout simplement parce qu'on ne sait pas que ça change quelque chose.

C'est exactement le genre de chose qu'on attend d'un agent : **qu'il aille chercher une information qu'on ne lui a pas demandée, et qu'il nous la signale.**

Le principe est simple, et la mécanique est déjà écrite pour toi dans la cellule suivante :

1. l'agent compose le voyage pour la date demandée, comme à l'Étape 3
2. il recommence pour chaque jour voisin (la fonction `dates_voisines` que je t'ai donnée) et garde ceux qui aboutissent, dans une liste `alternatives`
3. il compare, et si un jour voisin fait nettement mieux, il l'écrit dans le journal

Attention à un point de conception : **il ne déplace jamais tes dates tout seul.** Il te rend le voyage que tu as demandé, et il te signale l'occasion. Décider reste ton rôle, exactement comme pour la réservation qu'on verra juste après.

Reste une question, et c'est la seule vraiment intéressante : **comment décider qu'un jour voisin est « meilleur » ?**

Le réflexe serait de comparer les prix. C'est un piège, et il vaut la peine que tu comprennes pourquoi. Ton agent s'arrête dès qu'il passe sous le budget : quel que soit le jour, il finit donc toujours *juste* sous la barre. Comparer les totaux, ce serait comparer des prix presque identiques et ne rien voir.

Ce qui change réellement d'un jour à l'autre, ce n'est pas ce que le voyage coûte, c'est **ce qu'il reste dedans**. Un vol moins cher, c'est une sortie de plus qu'on n'a pas eu à sacrifier. On compare donc d'abord le nombre d'activités conservées, et le prix seulement en cas d'égalité.

En Python, ça s'écrit avec une clé à deux niveaux : `key=lambda v: (len(v["activites"]), -v["prix_total"])`. Le tuple est comparé élément par élément, donc le nombre de sorties prime, et le signe moins devant le prix veut dire « et à sorties égales, le moins cher gagne ». C'est une écriture qui te resservira souvent.

Il te reste trois lignes :

1. `meilleure` : parmi la liste `alternatives`, le voyage qui gagne selon cette clé. Attention, c'est un `max`, pas un `min`
2. `sorties_en_plus` : combien d'activités `meilleure` garde de plus que `voyage`
3. `economie` : combien d'euros `meilleure` coûte de moins que `voyage`

In [ ]:
def planifier(demande, conn, brochures, encodeur):
    """
    L'agent complet : il compose le voyage demandé, puis explore les jours voisins.

    Arguments :
    demande -- un dictionnaire avec destination, date_depart, nuits, voyageurs, budget_max, envie
    conn, brochures, encodeur -- les sources de données de l'agent

    Retourne :
    voyage -- le voyage pour la date demandée, ou None si rien n'y rentre
    journal -- les ajustements de l'agent, et le bon plan qu'il a repéré ailleurs
    """
    print(f"[agent] {demande['destination']}, {demande['nuits']} nuits, "
          f"{demande['voyageurs']} voyageur(s), budget {demande['budget_max']:.0f} EUR par personne")

    voyage, journal = essayer_une_date(demande, conn, brochures, encodeur)

    # On rejoue exactement le même raisonnement pour chaque jour voisin. Seul le prix du
    # vol change d'un jour à l'autre : les hôtels et les activités, eux, ne bougent pas.
    alternatives = []
    for autre_jour in dates_voisines(demande["date_depart"]):
        candidat, _ = essayer_une_date(dict(demande, date_depart=autre_jour), conn, brochures, encodeur)
        if candidat is not None:
            alternatives.append(candidat)

    if not alternatives:
        return voyage, journal

    if voyage is None:
        # Rien ne rentrait le jour demandé : on signale le jour voisin le moins cher.
        secours = min(alternatives, key=lambda v: v["prix_total"])
        jour = datetime.strptime(secours["date_depart"], "%Y-%m-%d").strftime("%d/%m")
        journal.append(f"en revanche, en partant le {jour}, un voyage à "
                       f"{secours['prix_total']:.0f} EUR par personne devenait possible")
        return voyage, journal

    ### START CODE HERE ###

    meilleure = # (1) le meilleur jour voisin
    sorties_en_plus = # (2) les sorties gagnées
    economie = # (3) les euros gagnés

    ### END CODE HERE ###

    jour = datetime.strptime(meilleure["date_depart"], "%Y-%m-%d").strftime("%d/%m")
    if sorties_en_plus > 0:
        journal.append(f"au passage, en partant le {jour} vous gardiez "
                       f"{len(meilleure['activites'])} sorties au lieu de {len(voyage['activites'])}, "
                       f"pour {meilleure['prix_total']:.0f} EUR")
    elif economie >= 10:
        journal.append(f"au passage, en partant le {jour} le même programme revenait à "
                       f"{meilleure['prix_total']:.0f} EUR, soit {economie:.0f} EUR de moins par personne")

    return voyage, journal


voyage, journal = planifier(demande, conn, brochures, encodeur)
print()
afficher_voyage(voyage, journal)

print()
print(texte_proposition(voyage, journal))

assert voyage is not None, "Un voyage devrait être trouvé pour cette demande"
assert voyage["date_depart"] == demande["date_depart"], \
    "L'agent ne doit jamais déplacer les dates tout seul : il signale, il ne décide pas"
assert any("en partant le" in ligne for ligne in journal), \
    "L'agent devrait avoir repéré un jour voisin moins cher et l'avoir noté dans le journal"
print("\nExercice validé !")

Relis le journal affiché au-dessus, c'est lui la vraie nouveauté. L'agent a essayé, constaté que ça dépassait, choisi quoi sacrifier, recommencé, exploré des jours que personne ne lui avait demandé d'explorer, et il est capable de te dire exactement ce qu'il a fait et pourquoi.

Et remarque une chose : à aucun moment un modèle de langage n'est intervenu. Cette explication est produite par ton code, donc elle est toujours exacte. Un modèle génératif placé ici n'aurait rien apporté, si ce n'est le risque de raconter joliment une décision qu'il n'a pas prise.

Il reste deux cas à regarder, et le premier est mon préféré de tout le projet : celui où la date demandée ne passe pas, mais où un jour voisin, si.

In [ ]:
serre = dict(demande, budget_max=660)   # trop juste pour le 12 août, mais pas pour tous les jours
voyage_serre, journal_serre = planifier(serre, conn, brochures, encodeur)
print()
afficher_voyage(voyage_serre, journal_serre)

assert voyage_serre is None, "Avec ce budget, rien ne doit rentrer le jour demandé"
assert any("devenait possible" in ligne for ligne in journal_serre), \
    "L'agent devrait avoir trouvé un jour voisin où le voyage tient"
print("\nPersonne ne lui avait demandé de regarder les autres jours. C'est ça, un agent.")

Voilà la phrase que ton code vient de produire tout seul : *« aucun voyage possible le 12 août, mais en partant le 10, oui »*. Le voyageur n'avait pas posé la question. Il ne savait même pas qu'il y avait une question à poser.

Retiens le raisonnement, parce qu'il est réutilisable partout : **quand une recherche échoue, l'élargir d'un cran coûte presque rien et rapporte beaucoup.** Ici, ça a coûté quatre requêtes SQL de plus.

Reste le dernier cas : celui où il n'y a vraiment aucune solution, nulle part.

In [ ]:
impossible = dict(demande, budget_max=200)
voyage_impossible, journal_impossible = planifier(impossible, conn, brochures, encodeur)
print()
afficher_voyage(voyage_impossible, journal_impossible)

assert voyage_impossible is None, "Avec 200 EUR, l'agent ne doit rien proposer du tout"
assert not any("devenait possible" in l for l in journal_impossible), \
    "Aucun jour ne tient dans 200 EUR : l'agent ne doit rien promettre"
print("\nUn agent qui invente une solution quand il n'y en a pas est un agent dangereux. Celui-ci sait dire non.")

# Partie 5 - Laisser l'Agent effectuer une réservation à notre place

Jusqu'ici, ton agent n'a fait que **lire / analyser** des données.

À présent, il est temps de le laisser prendre des actions !

Nous allons donc écrire une fonction `reserver` qui permet à l'agent de réserver un vol d'avion en inscrivant les infos de dates et de destinations dans la base de données, sous la table `reservations`


Dans la cellule suivante, **lis d'abord le garde-fou** que je t'ai écrit : si `confirme` vaut `False`, la fonction ressort immédiatement en expliquant ce qu'elle *aurait* réservé, sans avoir touché à la base. Cela permet à l'utilisateur de garder la main sur ce que l'agent fait (et ne pas réserver un avion sans que lui-même l'ait validé en amont)

Ton travail porte sur les deux lignes qui suivent :

1. écris la requête `INSERT` dans la variable `sql`. La table `reservations` existe déjà dans la base, et elle est vide. Ses dix colonnes sont `client`, `destination`, `date_depart`, `nuits`, `voyageurs`, `vol`, `hotel`, `activites`, `prix_total` et `reservee_le`. Comme à l'Étape 1, tu n'écris **aucune valeur** dans la requête : tu mets un `?` par colonne, et les vraies valeurs sont passées à part, dans le tuple que tu vois juste en dessous. Dix colonnes, donc dix `?`.
2. termine par `conn.commit()`. Sans cet appel, SQLite garde ton écriture en attente et l'oublie à la fermeture de la connexion : ta réservation n'existerait que dans ta tête.

In [ ]:
from datetime import datetime


def reserver(conn, voyage, client, confirme=False):
    """
    Enregistre le voyage dans la table reservations, mais seulement si le voyageur a confirmé.

    Arguments :
    conn -- une connexion ouverte vers voyages.db
    voyage -- le dictionnaire retourné par planifier
    client -- le nom du voyageur
    confirme -- doit valoir True pour que quoi que ce soit soit écrit

    Retourne :
    message -- ce qui s'est passé, en clair
    """
    if voyage is None:
        return "Rien à réserver : aucun voyage n'a été trouvé."

    # LE GARDE-FOU. Trois lignes, et c'est la partie la plus importante du fichier :
    # tant que le voyageur n'a pas dit oui, la fonction ressort sans rien avoir écrit.
    if not confirme:
        return (f"Rien n'a été réservé. Le voyage à {voyage['destination']} coûterait "
                f"{voyage['prix_total']:.0f} EUR par personne, soit "
                f"{voyage['prix_total'] * voyage['voyageurs']:.0f} EUR au total. "
                f"Il faut confirmer pour que la réservation soit enregistrée.")

    ### START CODE HERE ###

    sql = """
        
    """  # (1) dix colonnes, donc dix points d'interrogation

    ### END CODE HERE ###

    conn.execute(sql, (client, voyage["destination"], voyage["date_depart"], voyage["nuits"],
                       voyage["voyageurs"], voyage["vol"]["numero"], voyage["hotel"]["hotel"],
                       ", ".join(a["nom"] for a in voyage["activites"]),
                       voyage["prix_total"] * voyage["voyageurs"],
                       datetime.now().strftime("%Y-%m-%d %H:%M")))

    ### START CODE HERE ###

    # (2) enregistre le résultat dans la base de données

    ### END CODE HERE ###

    return (f"C'est réservé pour {client} : {voyage['destination']}, "
            f"{voyage['nuits']} nuits, {voyage['prix_total'] * voyage['voyageurs']:.0f} EUR au total.")


def combien_de_reservations():
    return len(requete(conn, "SELECT * FROM reservations"))


avant = combien_de_reservations()

print(reserver(conn, voyage, "Marc"))                      # sans confirmation
apres_refus = combien_de_reservations()

print()
print(reserver(conn, voyage, "Marc", confirme=True))       # avec confirmation
apres_accord = combien_de_reservations()

display(requete(conn, "SELECT client, destination, hotel, prix_total, reservee_le FROM reservations"))

assert apres_refus == avant, "Sans confirmation, l'agent ne doit RIEN ecrire dans la base"
assert apres_accord == avant + 1, "Avec confirmation, une ligne doit etre ajoutee"
print("Exercice validé ! Le premier appel n'a rien écrit, le second oui.")

# Partie 6 - La cerise sur le gâteau : l'application Web

Ton agent fonctionne, mais il vit dans un notebook. Et soyons honnêtes : Marc n'ouvrira jamais un notebook. Ce qu'il veut, c'est un site, avec des menus et des boutons. Comme cela : 

<div align="center">
<img src="./Streamlit.png" alt="ouvrir dans VSCode pour voir l'image" width="800"/>
</div>

C'est le moment d'installer la toute dernière librairie du cahier de vacances :

```bash
uv add streamlit
```

Streamlit transforme un script Python en application web, sans que tu écrives une seule ligne de HTML ou de JavaScript. Chaque élément d'interface est une simple fonction Python : `st.selectbox` affiche un menu déroulant, `st.date_input` un calendrier, `st.text_area` une zone de texte. À chaque fois que l'utilisateur touche un widget, Streamlit ré-exécute le script avec les nouvelles valeurs. C'est l'outil idéal pour donner une interface à un projet de data science.

J'ai préparé le fichier `app.py` dans le dossier du projet : toute la partie interface y est déjà écrite. Il n'y manque que le plus important, le branchement de ton agent, et c'est justement ton dernier exercice.

Mais avant ça, il faut que l'application puisse accéder à ton agent. La cellule ci-dessous fait quelque chose d'un peu magique : elle récupère le code source des fonctions que TU as écrites dans ce notebook, et les enregistre dans un fichier `agent.py`. C'est ce fichier que l'application importera. Autrement dit, ce qui tournera dans ton navigateur dans cinq minutes, c'est ton code à toi.

In [ ]:
import importlib
import inspect
from pathlib import Path

EN_TETE = (
    "# L'agent de voyage, écrit automatiquement depuis le notebook du Projet 07.\n"
    "from datetime import datetime, timedelta\n\n"
    "from utils import encoder, requete\n\n\n"
)

fonctions = [outil_vols, outil_activites, outil_hotels, chiffrer_voyage,
             dates_voisines, essayer_une_date, planifier, reserver]
corps = "\n\n".join(inspect.getsource(f) for f in fonctions)

Path("agent.py").write_text(EN_TETE + corps, encoding="utf-8")
print(f"agent.py écrit : {len(fonctions)} fonctions, {len((EN_TETE + corps).splitlines())} lignes")

import agent
agent = importlib.reload(agent)   # au cas où tu réexécutes cette cellule après une modification
assert hasattr(agent, "planifier"), "agent.py doit contenir la fonction planifier"
assert hasattr(agent, "reserver"), "agent.py doit contenir la fonction reserver"
print("Import vérifié !")

### Étape Finale : branche ton agent dans l'application

Ouvre maintenant le fichier `app.py` dans VSCode et parcours-le rapidement. Tu verras qu'il ressemble beaucoup à du Python normal : le haut du fichier construit le formulaire, le bas affiche le résultat. Remarque au passage le `with st.form(...)` : il regroupe tous les champs et fait en sorte que rien ne se déclenche tant que le bouton **Valider** n'a pas été cliqué. Sans lui, Streamlit relancerait la recherche à chaque lettre tapée.

Vers le milieu, tu trouveras une zone `### START CODE HERE ###` avec trois lignes à compléter. Ce sont les seules lignes qui concernent vraiment ton agent, tout le reste n'est que de la décoration.

À cet endroit du script, les widgets ont déjà été créés, et leurs valeurs t'attendent dans des variables :

| variable | ce qu'elle contient |
|---|---|
| `destination` | la ville choisie dans le menu, par exemple `"Madrid"` |
| `depart` | la date d'aller, extraite du calendrier. Attention : c'est un objet `date` de Python, pas un texte |
| `nuits`, `voyageurs`, `budget` | des nombres. `nuits` est calculé pour toi à partir des deux dates du calendrier |
| `envie` | le texte libre tapé par l'utilisateur |

Ton travail, en trois lignes :

1. convertir la date en texte au format de la base : `depart.strftime("%Y-%m-%d")` transforme l'objet date en `"2026-08-12"`
2. rassembler la demande dans le dictionnaire à 6 clés que ton agent attend : `destination`, `date_depart`, `nuits`, `voyageurs`, `budget_max` et `envie`
3. appeler `agent.planifier(demande, conn, brochures, encodeur)`, qui retourne le voyage et le journal

Pour t'entraîner sans quitter le notebook, la cellule suivante recrée la situation exacte de `app.py` : les mêmes noms de variables, avec des valeurs fixes. Écris tes trois lignes ici, vérifie qu'elles passent les tests, puis recopie-les dans `app.py` à la place des trois lignes marquées `None`.

Et si tu veux comprendre les widgets eux-mêmes, la documentation officielle de Streamlit est excellente : [docs.streamlit.io/develop/api-reference](https://docs.streamlit.io/develop/api-reference). Regarde en particulier `st.form`, `st.date_input` et `st.columns`, tu reconnaîtras tout ce que fait `app.py`.

In [ ]:
import datetime as dt

# Les variables, telles que app.py les reçoit de son formulaire
destination = "Barcelone"
depart = dt.date(2026, 8, 12)
nuits = 4
voyageurs = 2
budget = 750
envie = ENVIE

### START CODE HERE ###

date_texte = 
demande =   # (2) la demande complète, avec ses 6 clés
voyage, journal =  # (3) et on appelle l'agent

### END CODE HERE ###

print(f"Voyage trouvé : {voyage['hotel']['hotel']}, {voyage['prix_total']:.0f} EUR par personne")

assert demande["date_depart"] == "2026-08-12", "La date doit être convertie en texte, au format AAAA-MM-JJ"
assert set(demande) == {"destination", "date_depart", "nuits", "voyageurs", "budget_max", "envie"}, \
    "La demande doit contenir exactement les 6 clés attendues par l'agent"
assert isinstance(journal, list), "planifier retourne deux choses : le voyage, puis le journal (une liste)"
assert voyage is not None and voyage["prix_total"] <= budget, "Le voyage doit tenir dans le budget"
print("Exercice validé ! Recopie maintenant ces trois lignes dans app.py, à la place des trois None.")

### Lance ton application

Il ne reste qu'à ouvrir un terminal, te placer dans le dossier `Projet_07`, et lancer :

```bash
uv run streamlit run app.py
```

Ton navigateur s'ouvre. En haut, le formulaire : la destination, les dates, le budget, et la zone de texte pour décrire ce que tu cherches. Tu cliques sur **Valider**, et en dessous apparaît le voyage composé par ton agent, avec la liste de ce qu'il a dû sacrifier pour tenir le budget. Un bouton te permet même de réserver, après confirmation.

Si tu vois le message « Ton agent n'est pas encore branché », c'est que les trois lignes de l'Étape 6 n'ont pas encore été recopiées dans `app.py`.

Amuse-toi deux minutes : baisse le budget et regarde l'agent sacrifier les activités une par une. Change l'envie en « ambiance festive » et regarde l'hôtel changer. Change de ville et découvre d'autres programmes.

C'est ta première application. Elle tourne sur ta machine, avec ton agent dedans.


FÉLICITATIONS !

---
# Epilogue

Quelques jours plus tard, tu rentres enfin chez toi.

La valise n'est pas encore défaite. Par la fenêtre, la lumière a déjà changé : celle de fin août, plus basse, plus dorée. Et en rangeant ton ordinateur, tu repenses à tout ce que tu as vécu.

Un directeur d'aéroport qui t'a fait confiance sur un coin de table. Léa et ses cent vingt villages. Marc, qui ne voulait surtout pas qu'on raconte n'importe quoi à ses clients. Ta nièce, qui connaît maintenant le nom de tous les poissons. Bruno, qui garde probablement encore ta glace au frais (ahah)

Début juillet, c'étaient des inconnus. Aujourd'hui, ce sont des amis.

C'est peut-être ça, le vrai bilan de cet été : on commence avec des projets, on finit avec des gens.

Tout est parfait. L'été se termine exactement comme il devait se terminer.

...

Et puis ton téléphone vibre.

Cette fois-ci, il ne s'agit pas d'un nouveau projet, mais d'un email de `Guillaume - Machine Learnia`.

---

> # Merci ...
>
>Hello, ici Guillaume !
>
>Si vous recevez ce message, c'est que vous êtes allé au bout du Cahier de Vacances.
>
>
>Alors avant toute chose : merci. Sincèrement.
>
>Vous ne savez pas à quel point cet été m'a rendu heureux. Vos commentaires chaque dimanche, vos projets partagés, vos déblocages racontés... En sept ans de Machine Learnia, je n'avais jamais vécu ça.
>
>C'est pour ça que je ne veux pas que ça s'arrête là. À la rentrée, je vous propose qu'on se réunisse, tous : **trois soirées en direct, gratuites, les 1er, 3 et 5 septembre à 19h.**
>
>Parce qu'il faut que je vous dise une chose. Cet été, vous n'étiez pas seul : vous étiez plus de 4 000 à réaliser ces différents projets, chacun de son côté. Et vous, vous êtes allé au bout. L'immense majorité des gens qui « se mettent à la data » ne terminent jamais un seul projet. Ils s'arrêtent au premier obstacle. Vous, vous avez continué, dimanche après dimanche. Cette régularité, c'est **LA qualité qui fait les bons data scientists**. Tout le reste s'apprend.
>
>Et justement... le cahier vous a appris à réussir des projets préparés, avec les données prêtes et les exercices balisés. Dans le métier, personne ne prépare le notebook. Poser le problème, choisir l'approche, défendre ses choix : c'est un autre terrain, le plus recherché du marché. Et c'est exactement ce qu'on va explorer ensemble à la rentrée, en construisant en direct.
>
>Si vous avez aimé réaliser tout ces projets, ce qui arrive va vous plaire encore plus.
>
>D'ici là : gardez votre application sous la main (elle va resservir), et venez me dire en commentaire quel projet a été votre préféré de l'été. Je lis tout.
>
>L'invitation officielle, avec le lien d'inscription, arrive dans quelques jours.
>
>À très vite,
>
>Guillaume - Machine Learnia ✌️

---

Tu ouvres ton agenda, et tu notes trois dates.

L'été est peut-être fini...

... Mais l'histoire ne fait que commencer ! :)


<div align="center">
<img src="./Epilogue.png" alt="ouvrir dans VSCode pour voir l'image" width="1000"/>
</div>
